In [ ]:
!pip install pyDOE

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from pyDOE import lhs

In [ ]:
plt.rcParams.update({'font.size': 14})
tf.keras.backend.set_floatx('float64')

In [ ]:
class SumBranches(tf.keras.layers.Layer):
  def __init__(self, name=None, **kwargs):
    super(SumBranches, self).__init__(name=name, **kwargs)

    self.branch_weight = self.add_weight(
        shape=(1, ), initializer='ones', trainable=True
        )

    self.trunk_weight = self.add_weight(
        shape=(1, ), initializer='ones', trainable=True
        )

  def call(self, x):

    out = tf.reduce_sum(tf.add(self.branch_weight*x[0], 
                               self.trunk_weight*x[1]), axis=1)
    out = tf.expand_dims(out, axis=1)
    return out


class CombineBranches(tf.keras.layers.Layer):
  def __init__(self):
    super(CombineBranches, self).__init__()

  def call(self, inputs):
    x, y = inputs[0], inputs[1]
    out = tf.einsum('ij,kj->ik', x, y)
    return out


class PeriodicBCs(tf.keras.layers.Layer):
  def __init__(self, name=None, **kwargs):
    super(PeriodicBCs, self).__init__(name=name, **kwargs)

  def call(self, inputs):
    return tf.concat((tf.cos(inputs), tf.sin(inputs)), axis=1)


# Define the normalization layer
class Normalize(tf.keras.layers.Layer):
  def __init__(self, xmin, xmax, name=None, **kwargs):
    super(Normalize, self).__init__(name=name)
    self.xmin = xmin
    self.xmax = xmax
    super(Normalize, self).__init__(**kwargs)

  def call(self, inputs):
    return 2.0*(inputs-self.xmin)/(self.xmax-self.xmin)-1.0

  def get_config(self):
    config = super(Normalize, self).get_config()
    config.update({'xmin': self.xmin, 'xmax': self.xmax})
    return config

In [ ]:
# Architecture
n_units = 20
n_layers = 2
p = 50

# def transformer_net(inp, out_shape):

#   u = tf.keras.layers.Dense(n_units, activation='tanh')(inp)
#   v = tf.keras.layers.Dense(n_units, activation='tanh')(inp)

#   h = tf.keras.layers.Dense(n_units, activation='tanh')(inp)
#   for i in range(n_layers-1):
#     z = tf.keras.layers.Dense(n_units, activation='tanh')(h)
#     h = tf.keras.layers.Add()([tf.keras.layers.Multiply()([1-z,u]),
#                               tf.keras.layers.Multiply()([z,v])])
#   out = tf.keras.layers.Dense(out_shape, activation='linear', use_bias=False)(h)
#   return out

def transformer_net(inp, out_shape):

  h = tf.keras.layers.Dense(n_units, activation='tanh')(inp)
  for i in range(n_layers-1):
    h = tf.keras.layers.Dense(n_units, activation='tanh')(h)
  out = tf.keras.layers.Dense(out_shape, activation='linear')(h)
  return out

# Branch net
u_sample = 100

input_u = tf.keras.layers.Input(shape=(u_sample, ))
branch_net = transformer_net(input_u, p)

# Trunk net
input_t = tf.keras.layers.Input(shape=(1,))
input_x = tf.keras.layers.Input(shape=(1,))
periodic_x = PeriodicBCs()(input_x)

input_tx = tf.keras.layers.Concatenate()([input_t, periodic_x])
trunk_net = transformer_net(input_tx, p)

# DeepOnet
out = CombineBranches()([branch_net, trunk_net])

# out = tf.keras.layers.Reshape((-1,1,))(out)
# out = tf.keras.layers.Dense(n_units, activation='tanh')(out)
# out = tf.keras.layers.Dense(n_units, activation='tanh')(out)
# out = tf.keras.layers.Dense(1, activation='tanh')(out)
# out = tf.keras.layers.Reshape((-1,))(out)

model = tf.keras.models.Model([input_t, input_x, input_u], out)
model.summary()

In [ ]:
tf.keras.utils.plot_model(model)

In [ ]:
# Boundaries of the computational domain
xleft, xright = -np.pi, np.pi
t0, tfinal = 0, 1

In [ ]:
def u0(x, waves, amplitudes):

  u = 1/(2*np.pi)
  for i in range(waves):
    u += amplitudes[i]*np.sin((i+1)*x)
  return u/np.max(u)

In [ ]:
# Wave speed
C = 1

In [ ]:
def defineCollocationPoints(t_bdry, x_bdry, N_pde=10000, N_u=100):

  # Convert input to numpy array
  t_bdry, x_bdry = np.array(t_bdry), np.array(x_bdry)
  
  # Uniform random sampling for PDE points
  tx_min = np.array([t_bdry[0], x_bdry[0]])
  tx_max = np.array([t_bdry[1], x_bdry[1]])  
  pde_points = tx_min + (tx_max - tx_min)*lhs(2, N_pde)
  t_pde = pde_points[:, 0] 
  x_pde = pde_points[:, 1]
  pdes = np.column_stack([t_pde, x_pde])
  
  # Initial data sample
  x_init = np.linspace(x_bdry[0], x_bdry[1], u_sample)
  t_init = t_bdry[0]+0*x_init
  inits = np.column_stack([t_init, x_init])
  
  # Initial function sample
  us = []
  for i in range(N_u):
    waves = np.random.randint(1, 5)
    amplitudes = np.random.randn(waves)
    us.append(u0(x_init, waves, amplitudes))
  us = np.row_stack(us)

  return (pdes, inits, us)

In [ ]:
N_pde, N_u = 10000, 1000
(pdes, inits, us) = defineCollocationPoints([t0, tfinal], [xleft, xright], 
                                            N_pde, N_u)

In [ ]:
plt.plot(us[0,:])

In [ ]:
@tf.function
def compute_gradient(t, x, u):
  with tf.GradientTape(persistent=True) as tape2:
    tape2.watch(t), tape2.watch(x), tape2.watch(u)
    u_pred = []
    for i in range(u.shape[0]):
      u_pred.append(model([t, x, tf.expand_dims(u[i,:], axis=0)]))
    
  dudt, dudx = [], []
  for i in range(u.shape[0]):
    [ut, ux] = tape2.gradient(u_pred[i], [t, x])
    dudt.append(ut)
    dudx.append(ux)
  
  return [tf.convert_to_tensor(dudt), tf.convert_to_tensor(dudx)]

In [ ]:
# @tf.function
def train_network(pdes, inits, us, model):

    # PDE points
    t_pde, x_pde = pdes[:,:1], pdes[:,1:2]

    # Initial value points
    t_init, x_init = inits[:,:1], inits[:,1:2]

    # Outer gradient for tuning network parameters
    with tf.GradientTape(persistent=True) as tape:

      # Compute the gradients
      dudt, dudx = compute_gradient(t_pde, x_pde, us)
      
      # Solve the linear advection equation
      eqn = dudt + C*dudx

      # Define the PDE loss  
      PDEloss = tf.reduce_mean(tf.square(eqn))

      # Define the IVP loss
      u_init_pred = model([t_init, x_init, us])
      ICloss = tf.reduce_mean(tf.square(us - u_init_pred))

      # Total loss
      loss = PDEloss + ICloss

    # Compute loss gradients
    grads = tape.gradient(loss, model.trainable_variables)

    return PDEloss, ICloss, grads, l

In [ ]:
def PINNtrain(pde_points, inits, us, u0, epochs=10000):

  bs_us = 10

  ds_init = tf.data.Dataset.from_tensor_slices(inits)
  ds_pde = tf.data.Dataset.from_tensor_slices(pde_points)
  ds_u = tf.data.Dataset.from_tensor_slices(us)

  ds = tf.data.Dataset.zip((ds_pde, ds_init, ds_u))
  ds = ds.cache().shuffle(N_u).batch(bs_us)
  ds = ds.prefetch(tf.data.experimental.AUTOTUNE)

  # Generate a new model
  lr_model = 1e-3
  opt = tf.keras.optimizers.Adam(lr_model)

  epoch_loss = np.zeros(epochs)
  nr_batches = 0

  # Main training loop
  for i in range(epochs):

    # Training for that epoch
    for (pdes, inits, us) in ds:

      # Train the network
      PDEloss, ICloss, grads, l = train_network(pdes, inits, us, model)

      # Gradient step
      opt.apply_gradients(zip(grads, model.trainable_variables))

      epoch_loss[i] += PDEloss + ICloss
      nr_batches += 1

    # Get total epoch loss
    epoch_loss[i] /= nr_batches
    nr_batches = 0

    if (np.mod(i, 100)==0):
      print("PDE loss, IV loss in {}th epoch: {: 6.4f}, {: 6.4f}.".format(i, PDEloss, ICloss))

  return epoch_loss

In [ ]:
epochs = 5000
loss = PINNtrain(pdes, inits, us, u0, epochs)

In [ ]:
plt.semilogy(range(0, epochs), loss)
plt.xlabel('epochs')
plt.ylabel('loss')
plt.savefig('loss.png')
plt.grid()

In [ ]:
# Sample a new random wave
waves = np.random.randint(1,5)
amplitudes = np.random.randn(waves)

In [ ]:
t0 = 0

# Grid where to evaluate the model
l, m = 50, 100
t = np.linspace(t0, tfinal, l)
x = np.linspace(xleft, xright, m)
T_p, X_p  = np.meshgrid(t, x, indexing='ij')

tt = np.expand_dims(T_p.flatten(), axis=1)
xx = np.expand_dims(X_p.flatten(), axis=1)

x_points = np.linspace(xleft, xright, u_sample)
uu = np.expand_dims(u0(x_points, waves, amplitudes), axis=0)

u = model([tt, xx, uu])
u = np.reshape(u, (l, m))

In [ ]:
plt.figure(figsize=(21,5))
plt.subplot(131)
plt.contourf(X_p, T_p, u, 100, cmap=plt.cm.coolwarm)
plt.title('Neural network solution')
plt.xlabel('x')
plt.ylabel('t')
plt.colorbar()

plt.subplot(132)
u_true = u0(X_p-C*T_p, waves, amplitudes)
plt.contourf(X_p, T_p, u_true, 100, cmap=plt.cm.coolwarm)
plt.title('True solution')
plt.xlabel('x')
plt.ylabel('t')
plt.colorbar()

plt.subplot(133)
plt.contourf(X_p, T_p, u-u_true, 100, cmap=plt.cm.coolwarm)
plt.xlabel('x')
plt.ylabel('t')
plt.title("Difference NN solution and true solution")
plt.colorbar()
plt.show()

In [ ]:
# Compute total mass
mass = u[:,:-1].sum(axis=1)
plt.plot(np.linspace(t0,tfinal,l), (mass-mass[0])/mass[0])
plt.xlabel('t')
plt.ylabel('Relative mass error')
plt.grid()